In [ ]:
!pip install -U transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
checkpoint = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("code-search-net/code_search_net")

In [ ]:
if tokenizer.pad_token == "none":
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
dataset["train"].column_names

In [ ]:
def tokenizer_in(example):
    return tokenizer(
        example["func_code_string"],
        max_length=256,
        truncation=True
    )

In [ ]:
small_dataset = dataset["train"].select(range(1000))

tokens = small_dataset.map(
    tokenizer_in,
    batched=True
)

In [ ]:
!pip install -U peft

In [ ]:
!pip install -U torchao

In [ ]:
from peft import LoraConfig, get_peft_model , TaskType

In [ ]:
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 8,
    lora_alpha = 16,
    target_modules = ["q_proj", "v_proj"],
    lora_dropout = 0.05,
    bias = "none"
)

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

train_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,

    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_torch",

    logging_steps=10,
    logging_strategy="steps",
    report_to="none",

    save_strategy="epoch",
    save_total_limit=2,
    seed=42
)

`taking all parameters as low as possible to decrease the time taken for training the model`

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=train_arguments,
    train_dataset=tokens,
    data_collator = data_collator
)
trainer.train()

In [ ]:
model.save_pretrained("./lora_adapter")
tokenizer.save_pretrained("./lora_adapter")

In [ ]:
import torch

prompt = "Write a Python function to check whether a number is prime."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)